# Marint Naturkart hard/soft/mixture Labelling

Notebook used to setup classes for training of bunntype classifier. 


In [1]:
import geopandas as gpd
import pandas as pd

import subkart

# Labelling

Use [bunnsedimenter](https://kartkatalog.geonorge.no/metadata/79f0f17d-9f62-456d-b1a9-2a8c754c51c4)dataset and NiN-LM from https://static.ngu.no/Mareano/Kornstorrelse.html to map into soft, hard, and mixed.

In [ ]:
# https://static.ngu.no/Mareano/Kornstorrelse-NiN-LM.html
# also see https://static.ngu.no/Mareano/Kornstorrelse-NiN-LM.html  DK_EFGY hard while, DK_ABCD soft and DK_0 mixture
HARD_BOTTOM_LM_DK = ["DK_EFGY"]
SOFT_BOTTOM_LM_DK = ["DK_AB", "DK_C", "DK_D"]
MIXED_BOTTOM_LM_DK = ["DK_0"]

In [3]:
df_kornstr = pd.read_html("https://static.ngu.no/Mareano/Kornstorrelse-NiN-LM.html", skiprows=0)[1]

In [4]:
SOFT_BOTTOM_TYPES = {
    row["Kornstørrelse SOSI-kode"]: row["Kornstørrelse SOSI-navn"]
    for _, row in df_kornstr[df_kornstr["Dominerende kornstørrelse LM-DK"].isin(SOFT_BOTTOM_LM_DK)].iterrows()
}
HARD_BOTTOM_TYPES = {
    row["Kornstørrelse SOSI-kode"]: row["Kornstørrelse SOSI-navn"]
    for _, row in df_kornstr[df_kornstr["Dominerende kornstørrelse LM-DK"].isin(HARD_BOTTOM_LM_DK)].iterrows()
}
MIXTURE_BOTTOM_TYPES = {
    row["Kornstørrelse SOSI-kode"]: row["Kornstørrelse SOSI-navn"]
    for _, row in df_kornstr[df_kornstr["Dominerende kornstørrelse LM-DK"].isin(MIXED_BOTTOM_LM_DK)].iterrows()
}


df_types = pd.concat(
    {
        f"Løsbunn[ {', '.join(SOFT_BOTTOM_LM_DK)} ]": pd.DataFrame(
            list(SOFT_BOTTOM_TYPES.items()), columns=["SOSI-kode", "SOSI-navn"]
        ),
        f"Hardbunn[ {', '.join(HARD_BOTTOM_LM_DK)} ]": pd.DataFrame(
            list(HARD_BOTTOM_TYPES.items()), columns=["SOSI-kode", "SOSI-navn"]
        ),
        f"Blanding[ {', '.join(MIXED_BOTTOM_LM_DK)} ]": pd.DataFrame(
            list(MIXTURE_BOTTOM_TYPES.items()), columns=["SOSI-kode", "SOSI-navn"]
        ),
    },
    axis=1,
)

df_types.fillna("")

Løsbunn[ DK_AB, DK_C, DK_D ]                                  \
                      SOSI-kode                       SOSI-navn   
0                            10                            Leir   
1                            15                   Organisk slam   
2                            20                            Slam   
3                            21  Slam med blokker av sedimenter   
4                            30                 Sandholdig leir   
5                            40                 Sandholdig slam   
6                            50                            Silt   
7                            60                 Sandholdig silt   
8                            70                 Leirholdig sand   
9                            80                 Slamholdig sand   
10                           90                 Siltholdig sand   
11                           95                        Fin sand   
12                          100                            Sand   
13                          105                       Grov sand   
14                          110                 Grusholdig slam   
15                          115      Grusholdig sandholdig slam   
16                          120      Grusholdig slamholdig sand   
17                          130                 Grusholdig sand   
18                          140                 Slamholdig grus   
19                          150      Slamholdig sandholdig grus   
20                          160                 Sandholdig grus   
21                          170                            Grus   
22                          174                   Grus og stein   
23                          175            Grus, stein og blokk   

   Hardbunn[ DK_EFGY ]                                                     \
             SOSI-kode                                          SOSI-navn   
0                180.0                                     Stein og blokk   
1                300.0       Harde sedimenter eller sedimentære bergarter   
2                  1.0  Tynt eller usammenhengende sedimentdekke over ...   
3                  5.0                                         Bart fjell   
4                                                                           
5                                                                           
6                                                                           
7                                                                           
8                                                                           
9                                                                           
10                                                                          
11                                                                          
12                                                                          
13                                                                          
14                                                                          
15                                                                          
16                                                                          
17                                                                          
18                                                                          
19                                                                          
20                                                                          
21                                                                          
22                                                                          
23                                                                          

   Blanding[ DK_0 ]                                         
          SOSI-kode                              SOSI-navn  
0             185.0                    Sand, grus og stein  
1             190.0                          Sand og blokk  
2            

In [3]:
gdf_bunn = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/input/ngu_sediment/BunnsedimentKornstorDetalj.geo.parquet")

In [6]:
def to_bunn_type(kornstorrelse: int) -> str:
    if kornstorrelse in SOFT_BOTTOM_TYPES:
        return "løsbunn"
    elif kornstorrelse in HARD_BOTTOM_TYPES:
        return "fastbunn"
    elif kornstorrelse in MIXTURE_BOTTOM_TYPES:
        return "blanding"

In [7]:
gdf_bunn["BunnType"] = gdf_bunn["sedKornstørrelse"].map(to_bunn_type)

In [8]:
fname = subkart.utils.to_filename("nisjedata-substrat-klassifisering", "norge", "latest", gdf_bunn.crs.to_epsg())

gdf_bunn.to_file(f"{fname}.geojson", driver="GeoJSON")
gdf_bunn.to_parquet(f"{fname}.geo.parquet", compression="snappy")

subkart.utils.to_postgis(gdf_bunn, fname)

Table nisjedata-substrat-klassifisering_norge_latest uploaded to PostGIS.
